In [2]:
import os
from typing import Optional, List
import pandas as pd

In [22]:
# the files should always be stacked vertically and to ensure that the files need to have same headers. if the files dont have same headers raise an error saying that the files dont have same headers. the only case that the dataframes should be concated even if the header is not the same is that the data doesnt have properly defined headers (the headers are numbers only) in that case you can give change the feature names to f_{i} that i represents the feature number. also this is the case and headers dont exist (are numbers) this should be the case for all that are being concated. so if in any file the headers did not match the initial pattern an error should be raised. so we have a couple of cases. either all files have same headers and can be concated properly, the second case is that none of them have headers (all headers are numbers) but the number of headeres is the same which you can all headers f_{i}, the other case is that it is a mixed condition where some files in between either have different sizes or some have headers while the other dont which still should raise error

In [23]:
# def load_data(path_to_data: str, file_names: Optional[List[str]] = []):
#     data = pd.DataFrame()
    
#     # Check if the path is a file
#     if os.path.isfile(path_to_data):
#         if path_to_data.endswith('.csv') and (not file_names or os.path.basename(path_to_data) in file_names):
#             data = pd.read_csv(path_to_data)
#     # If the path is a directory
#     elif os.path.isdir(path_to_data):
#         for root, _, files in os.walk(path_to_data):
#             for file in files:
#                 if file.endswith('.csv') and (not file_names or file in file_names):
#                     print(root, file)
#                     file_path = os.path.join(root, file)
#                     temp_data = pd.read_csv(file_path)
#                     data = pd.concat([data, temp_data], ignore_index=True)
#     else:
#         raise ValueError(f"Invalid path: {path_to_data}")

#     return data

In [5]:
def load_data(path_to_data: str, file_names: Optional[List[str]] = []):
    data = pd.DataFrame()
    headers = None
    is_numeric_headers = None

    # Check if the path is a file
    if os.path.isfile(path_to_data):
        if path_to_data.endswith('.csv') and (not file_names or os.path.basename(path_to_data) in file_names):
            temp_data = pd.read_csv(path_to_data, header=None)
            headers = temp_data.iloc[0].tolist() if temp_data.iloc[0].dtype == 'object' else None
            is_numeric_headers = headers is None
            if is_numeric_headers:
                temp_data.columns = [f"f_{i}" for i in range(temp_data.shape[1])]
            else:
                temp_data = pd.read_csv(path_to_data)
            temp_data['file_name'] = os.path.basename(path_to_data)
            data = temp_data
    # If the path is a directory
    elif os.path.isdir(path_to_data):
        for root, _, files in os.walk(path_to_data):
            for file in files:
                if file.endswith('.csv') and (not file_names or file in file_names):
                    file_path = os.path.join(root, file)
                    temp_data = pd.read_csv(file_path, header=None)
                    temp_headers = temp_data.iloc[0].tolist() if temp_data.iloc[0].dtype == 'object' else None
                    temp_is_numeric_headers = temp_headers is None

                    if headers is None:
                        headers = temp_headers
                        is_numeric_headers = temp_is_numeric_headers
                        if is_numeric_headers:
                            temp_data.columns = [f"f_{i}" for i in range(temp_data.shape[1])]
                        else:
                            temp_data = pd.read_csv(file_path)
                    else:
                        if is_numeric_headers != temp_is_numeric_headers:
                            raise ValueError("Mixed header conditions detected. Some files have headers while others do not.")
                        if is_numeric_headers:
                            if temp_data.shape[1] != len(headers):
                                raise ValueError("Files do not have the same number of columns.")
                            temp_data.columns = [f"f_{i}" for i in range(temp_data.shape[1])]
                        else:
                            if temp_headers != headers:
                                raise ValueError("Files do not have the same headers.")
                            temp_data = pd.read_csv(file_path)

                    temp_data['file_name'] = file  # Add file name as a column
                    data = pd.concat([data, temp_data], ignore_index=True)
    else:
        raise ValueError(f"Invalid path: {path_to_data}")

    return data

In [6]:
path_to_data = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded/cropped"
file_names = ['1_01_0_30fps_cropped.csv', '1_01_5_30fps_cropped.csv', '1_01_10_30fps_cropped.csv']
data = load_data(path_to_data, file_names)
print(data.shape)
data.head()

(54090, 129)


,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,f_9,...,f_119,f_120,f_121,f_122,f_123,f_124,f_125,f_126,f_127,file_name
0,0.800,0.204,-0.012,-0.106,0.617,1.213,-0.202,-0.954,0.643,1.125,...,-2.184,1.400,0.081,1.422,-0.338,-0.247,0.347,-1.379,-0.096,1_01_0_30fps_cropped.csv
1,0.749,0.102,0.031,-0.038,0.525,1.192,-0.209,-0.978,0.657,1.112,...,-2.193,1.268,0.128,1.417,-0.382,-0.171,0.400,-1.450,-0.151,1_01_0_30fps_cropped.csv
2,0.936,0.156,-0.100,-0.095,0.615,1.216,-0.367,-0.846,0.798,1.108,...,-2.140,1.362,-0.018,1.448,-0.275,-0.229,0.333,-1.260,-0.181,1_01_0_30fps_cropped.csv
3,0.765,0.078,0.253,0.019,0.630,1.214,-0.177,-0.902,0.871,0.846,...,-2.308,1.322,0.054,1.396,-0.515,-0.030,0.250,-1.481,-0.142,1_01_0_30fps_cropped.csv
4,0.544,-0.267,0.559,-0.202,0.133,1.539,-0.369,-0.660,0.573,1.245,...,-2.259,0.699,-0.688,1.350,-0.171,-0.180,0.234,-1.196,-0.252,1_01_0_30fps_cropped.csv


In [7]:
data

,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,f_9,...,f_119,f_120,f_121,f_122,f_123,f_124,f_125,f_126,f_127,file_name
0,0.800,0.204,-0.012,-0.106,0.617,1.213,-0.202,-0.954,0.643,1.125,...,-2.184,1.400,0.081,1.422,-0.338,-0.247,0.347,-1.379,-0.096,1_01_0_30fps_cropped.csv
1,0.749,0.102,0.031,-0.038,0.525,1.192,-0.209,-0.978,0.657,1.112,...,-2.193,1.268,0.128,1.417,-0.382,-0.171,0.400,-1.450,-0.151,1_01_0_30fps_cropped.csv
2,0.936,0.156,-0.100,-0.095,0.615,1.216,-0.367,-0.846,0.798,1.108,...,-2.140,1.362,-0.018,1.448,-0.275,-0.229,0.333,-1.260,-0.181,1_01_0_30fps_cropped.csv
3,0.765,0.078,0.253,0.019,0.630,1.214,-0.177,-0.902,0.871,0.846,...,-2.308,1.322,0.054,1.396,-0.515,-0.030,0.250,-1.481,-0.142,1_01_0_30fps_cropped.csv
4,0.544,-0.267,0.559,-0.202,0.133,1.539,-0.369,-0.660,0.573,1.245,...,-2.259,0.699,-0.688,1.350,-0.171,-0.180,0.234,-1.196,-0.252,1_01_0_30fps_cropped.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54085,1.453,0.991,-0.553,0.163,1.677,0.414,0.077,-1.858,0.583,0.924,...,-2.113,2.843,0.938,1.721,-1.181,0.313,-0.323,-0.450,-0.308,1_01_5_30fps_cropped.csv
54086,1.570,0.869,-0.676,0.122,1.538,0.648,0.022,-1.871,0.612,0.930,...,-2.116,2.833,0.781,1.671,-1.168,0.329,-0.266,-0.566,-0.374,1_01_5_30fps_cropped.csv
54087,1.514,0.897,-0.493,0.055,1.577,0.567,0.078,-1.891,0.503,0.888,...,-2.073,2.931,0.807,1.648,-1.265,0.348,-0.424,-0.669,-0.367,1_01_5_30fps_cropped.csv
54088,1.358,0.701,-0.695,0.171,1.590,0.627,0.029,-1.776,0.645,0.974,...,-2.099,2.802,0.928,1.781,-1.160,0.129,-0.339,-0.603,-0.405,1_01_5_30fps_cropped.csv


In [34]:
def process_dataset(dataset: pd.DataFrame, mapping: Optional[dict] = None) -> pd.DataFrame:
    # Ensure the 'file_name' column exists
    if 'file_name' not in dataset.columns:
        raise ValueError("The dataset must contain a 'file_name' column.")

    # Add the 'frame' column
    dataset['frame'] = dataset.groupby('file_name').cumcount()

    # Split 'file_name' to extract 'fold', 'subject_id', 'target', and 'type'
    split_columns = dataset['file_name'].str.split('_', expand=True)
    dataset['fold'] = split_columns[0].astype(int)
    dataset['subject_id'] = split_columns[1].astype(int).astype(str)
    dataset['target'] = split_columns[2]
    dataset['type'] = split_columns[4].str.replace('.csv', '', regex=False)

    # Handle the 'target' column
    if mapping:
        unique_targets = dataset['target'].unique()
        if not set(unique_targets).issubset(set(mapping.keys())):
            raise ValueError("Mapping does not include all unique target values.")
        dataset['target'] = dataset['target'].map(mapping).astype(float)
    else:
        dataset['target'] = dataset['target'].astype(float)

    # Reorder columns
    columns_order = ['file_name', 'type', 'frame', 'fold', 'subject_id', 'target']
    remaining_columns = [col for col in dataset.columns if col not in columns_order]
    dataset = dataset[columns_order + remaining_columns]

    return dataset

In [32]:
p_data = process_dataset(data, mapping={'0':0, '5':2, '10':1})

In [33]:
p_data['subject_id']

0        01
1        01
2        01
3        01
4        01
         ..
54085    01
54086    01
54087    01
54088    01
54089    01
Name: subject_id, Length: 54090, dtype: object

In [ ]:
from typing import Callable, List, Dict

def process_in_parallel(worker_function: Callable, worker_params_list: List[Dict], max_workers: int) -> None:
    logging.info("Processing in Parallel Mode ...")
    with multiprocessing.Pool(processes=max_workers, maxtasksperchild=1) as pool:
        pool.map(worker_function, worker_params_list)

def process_in_serial(worker_function: Callable, worker_params_list: List[Dict]) -> None:
    logging.info("Processing in Serialized Mode ...")
    for params in worker_params_list:
        worker_function(**params)

# Example usage
worker_function = process_single_video_wrapper if parallel else process_single_video
worker_params_list: List[Dict] = [
    {
        "video_file_path": video_file_paths[i],
        "input_folder_path": input_folder_path,
        "output_folder_path": output_folder_path,
        "start_second": start_second,
        "end_second": end_second,
        "embedding_model": embedding_model,
        "embedding_normalization": embedding_normalization,
        "use_tqdm": use_tqdm,
        "file_index": i,
    }
    for i in range(len(video_file_paths))
]

if parallel:
    process_in_parallel(worker_function, worker_params_list, max_workers)
else:
    process_in_serial(worker_function, worker_params_list)
